# 04 - Unsloth Fine-tuning

This notebook fine-tunes Gemma using Unsloth's QLoRA implementation for efficient training on Kaggle GPUs.

In [ ]:
!pip install -q unsloth
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from training.unsloth_trainer import UnslothTrainer, TrainingConfig, DataConfig, run_training_pipeline

SOURCE_LANG = "so"
TARGET_LANG = "sw"
DATA_DIR = Path(f'../data/synthetic/{SOURCE_LANG}-{TARGET_LANG}/validated')
OUTPUT_DIR = Path(f'../outputs/{SOURCE_LANG}-{TARGET_LANG}')

In [ ]:
# Training configuration
training_config = TrainingConfig(
    base_model="unsloth/gemma-2-9b-it-bnb-4bit",
    max_seq_length=2048,
    lora_r=16,
    lora_alpha=16,
    batch_size=2,
    gradient_accumulation_steps=4,
    num_epochs=3,
    learning_rate=2e-4,
    output_dir=str(OUTPUT_DIR)
)

data_config = DataConfig(
    source_lang=SOURCE_LANG,
    target_lang=TARGET_LANG,
    source_lang_name="Somali",
    target_lang_name="Swahili"
)

In [ ]:
# Initialize trainer
trainer = UnslothTrainer(training_config, data_config)

# Load model with LoRA
model, tokenizer = trainer.load_model()

In [ ]:
# Prepare dataset
dataset = trainer.prepare_dataset(
    source_file=str(DATA_DIR / f'synthetic_filtered.{SOURCE_LANG}'),
    target_file=str(DATA_DIR / f'synthetic_filtered.{TARGET_LANG}')
)

# Split for validation
split = dataset.train_test_split(test_size=0.05)
train_dataset = split['train']
eval_dataset = split['test']

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(eval_dataset)}")

In [ ]:
# Train
trainer.train(train_dataset, eval_dataset)

In [ ]:
# Save model
trainer.save_model(
    output_dir=str(OUTPUT_DIR / 'final'),
    save_merged=True,
    quantization='q4_k_m'
)

In [ ]:
# Generate Modelfile for Ollama
gguf_path = OUTPUT_DIR / 'final' / 'gguf' / f'daraja-{SOURCE_LANG}-{TARGET_LANG}.gguf'
trainer.generate_modelfile(
    gguf_path=str(gguf_path),
    output_path=str(OUTPUT_DIR / f'daraja-{SOURCE_LANG}-{TARGET_LANG}.Modelfile')
)

In [ ]:
print("=" * 50)
print("TRAINING COMPLETE")
print("=" * 50)
print(f"\nModel saved to: {OUTPUT_DIR}")
print(f"\nNext step: Run 05_evaluation.ipynb")